In [9]:
import os
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score
from collections import Counter
import hashlib

# =========================================================
# 🔥 OPTIMIZATION: 12 Threads & CUDA Setup
# =========================================================
torch.set_num_threads(12) # Menetapkan 12 threads CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PROCESSED_DIR = r'D:\S2\thesis\cond\project_ids\data\processed'

print(f"🚀 Perangkat: {device} | Threads CPU: {torch.get_num_threads()}")
print("🔍 LOADING DATA...")
data = joblib.load(os.path.join(PROCESSED_DIR, 'data_scratch.pkl'))

X_train = np.array(data['X_train'])
X_val   = np.array(data['X_val'])

# =========================================================
# 🔥 FIX: Proper Label Encoding
# =========================================================
all_classes = sorted(list(set(data['y_train'])))
label_map = {c: i for i, c in enumerate(all_classes)}

y_train = np.array([label_map[y] for y in data['y_train']])
y_val   = np.array([label_map[y] for y in data['y_val']])

num_classes = len(label_map)

print(f"Train size: {len(X_train):,}")
print(f"Val size  : {len(X_val):,}")
print(f"Num classes: {num_classes}")
print("="*70)


# =========================================================
# 1️⃣ DUPLICATE CHECK
# =========================================================
print("🔎 Checking duplicate rows between Train & Val...")

def row_hash(arr):
    return hashlib.md5(arr.tobytes()).hexdigest()

train_hashes = set(row_hash(row) for row in X_train)
val_hashes   = set(row_hash(row) for row in X_val)

intersection = train_hashes.intersection(val_hashes)

print(f"Duplicate rows across splits: {len(intersection)}")

if len(intersection) > 0:
    print("❌ WARNING: Data leakage detected (duplicate samples across splits)")
else:
    print("✅ No duplicate rows across splits")

print("="*70)


# =========================================================
# 2️⃣ INTERNAL DUPLICATE CHECK
# =========================================================
print("🔎 Checking internal duplicates...")

train_unique = len(train_hashes)
val_unique   = len(val_hashes)

print(f"Train duplicates: {len(X_train) - train_unique}")
print(f"Val duplicates  : {len(X_val) - val_unique}")

print("="*70)


# =========================================================
# 3️⃣ CLASS DISTRIBUTION CHECK
# =========================================================
print("📊 Checking class distribution...")

train_dist = Counter(y_train)
val_dist   = Counter(y_val)

print("Top 5 Train classes:", train_dist.most_common(5))
print("Top 5 Val classes  :", val_dist.most_common(5))

print("="*70)


# =========================================================
# 4️⃣ SHUFFLE LABEL TEST
# =========================================================
print("🧪 Running Shuffle Label Sanity Test...")

# Small subset for quick test
sample_size = 100000
X_sample = torch.FloatTensor(X_train[:sample_size]).to(device)
y_shuffled = np.random.permutation(y_train[:sample_size])
y_sample = torch.LongTensor(y_shuffled).to(device)

model = nn.Linear(X_train.shape[1], num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for _ in range(5):
    optimizer.zero_grad()
    outputs = model(X_sample)
    loss = criterion(outputs, y_sample)
    loss.backward()
    optimizer.step()

with torch.no_grad():
    preds = model(X_sample).argmax(1)
    acc = accuracy_score(y_sample.cpu(), preds.cpu())

print(f"Shuffle label accuracy: {acc:.4f}")
print(f"Expected random accuracy ≈ {1/num_classes:.4f}")

if acc > (1/num_classes) * 5:
    print("❌ WARNING: Possible leakage (model still learning shuffled labels)")
else:
    print("✅ Shuffle test passed")

print("="*70)


# =========================================================
# 5️⃣ FEATURE-LABEL CORRELATION CHECK
# =========================================================
print("🔎 Checking feature-label correlation...")

df = pd.DataFrame(X_train[:200000])
df['label'] = y_train[:200000]

corrs = df.corr()['label'].abs().sort_values(ascending=False)

print("Top 10 feature-label correlations:")
print(corrs.head(10))

if corrs.iloc[1] > 0.95:
    print("❌ WARNING: Extremely high correlation feature detected")
else:
    print("✅ No extreme feature-label correlation")

print("="*70)

print("🔐 Leakage Audit Complete")

🚀 Perangkat: cuda | Threads CPU: 12
🔍 LOADING DATA...
Train size: 4,285,924
Val size  : 1,071,482
Num classes: 34
🔎 Checking duplicate rows between Train & Val...
Duplicate rows across splits: 0
✅ No duplicate rows across splits
🔎 Checking internal duplicates...
Train duplicates: 0
Val duplicates  : 0
📊 Checking class distribution...
Top 5 Train classes: [(6, 661827), (14, 497624), (13, 412422), (8, 375601), (10, 373582)]
Top 5 Val classes  : [(6, 165457), (14, 124406), (13, 103106), (8, 93901), (10, 93396)]
🧪 Running Shuffle Label Sanity Test...
Shuffle label accuracy: 0.0683
Expected random accuracy ≈ 0.0294
✅ Shuffle test passed
🔎 Checking feature-label correlation...
Top 10 feature-label correlations:
label    1.000000
2        0.619580
28       0.488311
25       0.423861
32       0.357694
11       0.263627
10       0.235639
39       0.227192
7        0.187193
14       0.183467
Name: label, dtype: float64
✅ No extreme feature-label correlation
🔐 Leakage Audit Complete


In [8]:
import pandas as pd
import numpy as np
import joblib
import torch
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

# Konfigurasi
torch.set_num_threads(12)
PROCESSED_DIR = r'D:\S2\thesis\cond\project_ids\data\processed'
PATH_DATA = os.path.join(PROCESSED_DIR, 'train_cleaned.parquet')

print("⏳ Tahap Purifikasi: Loading Data Asli...")
df = pd.read_parquet(PATH_DATA)
initial_rows = len(df)

# 1. Hapus Duplikat Total (Internal & Cross-split nantinya)
print("🔎 Menghapus baris duplikat...")
df = df.drop_duplicates().reset_index(drop=True)
purified_rows = len(df)

print(f"✅ Berhasil menghapus {initial_rows - purified_rows:,} baris duplikat.")

# 2. Pisahkan Fitur dan Label
X = df.drop(columns=['label'])
y = df['label']

# 3. Stratified Split (Pastikan distribusi kelas terjaga)
print("⚖️ Melakukan Stratified Split ulang (80/20)...")
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 4. Scaling (Fit hanya di train!)
print("📏 Scaling fitur...")
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

# 5. Simpan Ulang
print(f"💾 Menyimpan data bersih ke {PROCESSED_DIR}...")
data_clean = {
    'X_train': X_train_scaled,
    'X_val': X_val_scaled,
    'y_train': y_train.values,
    'y_val': y_val.values,
    'classes': np.unique(y)
}

joblib.dump(scaler, os.path.join(PROCESSED_DIR, 'scaler_scratch.pkl'))
joblib.dump(data_clean, os.path.join(PROCESSED_DIR, 'data_scratch.pkl'))

print("-" * 50)
print(f"📊 DATA PURIFIED!")
print(f"Final Train size: {len(X_train_scaled):,}")
print(f"Final Val size  : {len(X_val_scaled):,}")
print("🔐 Kebocoran data (Leakage) resmi ditutup.")

⏳ Tahap Purifikasi: Loading Data Asli...
🔎 Menghapus baris duplikat...
✅ Berhasil menghapus 134,565 baris duplikat.
⚖️ Melakukan Stratified Split ulang (80/20)...
📏 Scaling fitur...
💾 Menyimpan data bersih ke D:\S2\thesis\cond\project_ids\data\processed...
--------------------------------------------------
📊 DATA PURIFIED!
Final Train size: 4,285,924
Final Val size  : 1,071,482
🔐 Kebocoran data (Leakage) resmi ditutup.


In [1]:
import os
import time
import joblib
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import f1_score
from collections import Counter
import pandas as pd

# =========================================================
# 1. Hardware & Path Setup
# =========================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(12)

PROCESSED_DIR = r'D:\S2\thesis\cond\project_ids\data\processed'
BATCH_SIZE = 4096
EPOCHS = 50
MAX_LR = 0.001
BASE_LR = 0.0001

# =========================================================
# 2. Residual Block
# =========================================================
class ResidualBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim),
            nn.ReLU(),
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim)
        )

    def forward(self, x):
        return torch.relu(x + self.net(x))

# =========================================================
# 3. Smart SLGRAE Model
# =========================================================
class Smart_SLGRAE_Latent32(nn.Module):
    def __init__(self, input_dim=44, latent_dim=32, num_classes=34):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256), nn.ReLU(), ResidualBlock(256),
            nn.Linear(256, 128), nn.ReLU(), ResidualBlock(128),
            nn.Linear(128, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.ReLU(),
            nn.Linear(128, 256), nn.ReLU(),
            nn.Linear(256, input_dim)
        )
        self.classifier = nn.Sequential(
            nn.Linear(latent_dim, 256), nn.BatchNorm1d(256), nn.ReLU(),
            nn.Dropout(0.4), ResidualBlock(256),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        latent = self.encoder(x)
        return self.classifier(latent), self.decoder(latent), latent

# =========================================================
# 4. Load Purified Data
# =========================================================
print("⏳ Loading purified data...")
data = joblib.load(os.path.join(PROCESSED_DIR, 'data_scratch.pkl'))
class_names = data['classes']

# Safe mapping for string or numeric labels
if isinstance(data['y_train'][0], str):
    label_map = {name: i for i, name in enumerate(class_names)}
    y_train_mapped = [label_map[l] for l in data['y_train']]
    y_val_mapped   = [label_map[l] for l in data['y_val']]
else:
    y_train_mapped = data['y_train']
    y_val_mapped   = data['y_val']

# Convert to DataFrames for leakage check
df_train = pd.DataFrame(data['X_train'])
df_val   = pd.DataFrame(data['X_val'])

# Initial leakage check
leakage = pd.merge(df_train, df_val, how='inner')
print(f"🔍 Initial duplicate rows between train & val: {len(leakage)}")

# DataLoaders
train_loader = DataLoader(
    TensorDataset(torch.FloatTensor(data['X_train']),
                  torch.LongTensor(y_train_mapped)),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True
)
val_loader = DataLoader(
    TensorDataset(torch.FloatTensor(data['X_val']),
                  torch.LongTensor(y_val_mapped)),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True
)

# =========================================================
# 5. Initialize Model, Optimizer, Scheduler, Loss
# =========================================================
model = Smart_SLGRAE_Latent32().to(device)
optimizer = optim.Adam(model.parameters(), lr=BASE_LR)
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=MAX_LR, steps_per_epoch=len(train_loader), epochs=EPOCHS
)

criterion_cls = nn.CrossEntropyLoss()
criterion_rec = nn.MSELoss()

# =========================================================
# 6. Training Loop with Leakage & Class Monitoring
# =========================================================
print(f"\n🔥 Training Start | Epochs: {EPOCHS} | Batch: {BATCH_SIZE} | Device: {device}")
print("-"*120)

best_val_loss = float("inf")

for epoch in range(1, EPOCHS + 1):
    start_time = time.time()

    # --- TRAINING PHASE ---
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0

    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad(set_to_none=True)

        y_pred, x_recon, _ = model(X_batch)
        l_cls = criterion_cls(y_pred, y_batch)
        l_rec = criterion_rec(x_recon, X_batch)
        loss = l_cls + 0.1*l_rec

        loss.backward()
        optimizer.step()
        scheduler.step()

        train_loss += loss.item()
        train_correct += (y_pred.argmax(1) == y_batch).sum().item()
        train_total += y_batch.size(0)

    # --- VALIDATION PHASE ---
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    all_preds, all_true = [], []

    with torch.no_grad():
        for X_val, y_val in val_loader:
            X_val, y_val = X_val.to(device), y_val.to(device)
            y_val_pred, x_val_recon, _ = model(X_val)
            v_loss = criterion_cls(y_val_pred, y_val) + 0.1*criterion_rec(x_val_recon, X_val)
            val_loss += v_loss.item()

            preds = y_val_pred.argmax(1)
            val_correct += (preds == y_val).sum().item()
            val_total += y_val.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_true.extend(y_val.cpu().numpy())

    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss   = val_loss / len(val_loader)
    val_acc        = 100*val_correct / val_total
    macro_f1       = f1_score(all_true, all_preds, average='macro')

    # --- Check for leakage in this epoch (optional) ---
    # Quick check: recombine val batch with train hash
    df_val_check = pd.DataFrame(data['X_val'])
    leakage_count = len(pd.merge(df_train, df_val_check, how='inner'))

    # --- Check class distribution ---
    train_class_dist = Counter(y_train_mapped)
    val_class_dist   = Counter(y_val_mapped)

    status = ""
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save({'model_state_dict': model.state_dict()}, 'best_baseline_model.pth')
        status = "⭐ Best"

    print(f"Ep [{epoch:02d}/{EPOCHS}] "
          f"LR: {optimizer.param_groups[0]['lr']:.6f} | "
          f"Tr_L: {avg_train_loss:.4f} | Val_L: {avg_val_loss:.4f} | "
          f"Val_Acc: {val_acc:.2f}% | F1: {macro_f1:.4f} | "
          f"Leak: {leakage_count} | "
          f"Train Class[Top5]: {train_class_dist.most_common(5)} | "
          f"Val Class[Top5]: {val_class_dist.most_common(5)} | "
          f"{time.time()-start_time:.1f}s {status}")

print("-"*120)
print(f"✅ Training Complete | Best Val Loss: {best_val_loss:.4f}")


⏳ Loading purified data...
🔍 Initial duplicate rows between train & val: 0

🔥 Training Start | Epochs: 50 | Batch: 4096 | Device: cuda
------------------------------------------------------------------------------------------------------------------------
Ep [01/50] LR: 0.000050 | Tr_L: 0.7799 | Val_L: 0.4620 | Val_Acc: 79.63% | F1: 0.4309 | Leak: 0 | Train Class[Top5]: [(6, 661827), (14, 497624), (13, 412422), (8, 375601), (10, 373582)] | Val Class[Top5]: [(6, 165457), (14, 124406), (13, 103106), (8, 93901), (10, 93396)] | 29.9s ⭐ Best
Ep [02/50] LR: 0.000082 | Tr_L: 0.3831 | Val_L: 0.7331 | Val_Acc: 74.59% | F1: 0.4796 | Leak: 0 | Train Class[Top5]: [(6, 661827), (14, 497624), (13, 412422), (8, 375601), (10, 373582)] | Val Class[Top5]: [(6, 165457), (14, 124406), (13, 103106), (8, 93901), (10, 93396)] | 32.6s 
Ep [03/50] LR: 0.000132 | Tr_L: 0.3144 | Val_L: 1.5769 | Val_Acc: 64.29% | F1: 0.4411 | Leak: 0 | Train Class[Top5]: [(6, 661827), (14, 497624), (13, 412422), (8, 375601), (10,

In [2]:
import os
import time
import joblib
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight
from collections import Counter
import pandas as pd

# =========================================================
# 1. Hardware & Path Setup
# =========================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(12)

PROCESSED_DIR = r'D:\S2\thesis\cond\project_ids\data\processed'
BATCH_SIZE = 4096
EPOCHS = 50
MAX_LR = 0.001
BASE_LR = 0.0001

# =========================================================
# 2. Arsitektur Model (Tetap SLGRAE)
# =========================================================
class ResidualBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim), nn.BatchNorm1d(dim), nn.ReLU(),
            nn.Linear(dim, dim), nn.BatchNorm1d(dim)
        )
    def forward(self, x):
        return torch.relu(x + self.net(x))

class Smart_SLGRAE_Latent32(nn.Module):
    def __init__(self, input_dim=44, latent_dim=32, num_classes=34):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256), nn.ReLU(), ResidualBlock(256),
            nn.Linear(256, 128), nn.ReLU(), ResidualBlock(128),
            nn.Linear(128, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.ReLU(),
            nn.Linear(128, 256), nn.ReLU(),
            nn.Linear(256, input_dim)
        )
        self.classifier = nn.Sequential(
            nn.Linear(latent_dim, 256), nn.BatchNorm1d(256), nn.ReLU(),
            nn.Dropout(0.4), ResidualBlock(256),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, num_classes)
        )
    def forward(self, x):
        latent = self.encoder(x)
        return self.classifier(latent), self.decoder(latent), latent

# =========================================================
# 3. Load Data & Calculate Class Weights (Imbalance-Aware)
# =========================================================
print("⏳ Loading purified data & calculating weights...")
data = joblib.load(os.path.join(PROCESSED_DIR, 'data_scratch.pkl'))
class_names = data['classes']
label_map = {name: i for i, name in enumerate(class_names)}

y_train_mapped = np.array([label_map[l] for l in data['y_train']])
y_val_mapped   = np.array([label_map[l] for l in data['y_val']])

# --- HITUNG BOBOT KELAS ---
weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train_mapped),
    y=y_train_mapped
)
class_weights = torch.FloatTensor(weights).to(device)

train_loader = DataLoader(
    TensorDataset(torch.FloatTensor(data['X_train']), torch.LongTensor(y_train_mapped)),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True
)
val_loader = DataLoader(
    TensorDataset(torch.FloatTensor(data['X_val']), torch.LongTensor(y_val_mapped)),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True
)

# Untuk pengecekan leakage (statis)
df_train = pd.DataFrame(data['X_train'])

# =========================================================
# 4. Initialize Model & Weighted Loss
# =========================================================
model = Smart_SLGRAE_Latent32().to(device)
optimizer = optim.Adam(model.parameters(), lr=BASE_LR)
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=MAX_LR, steps_per_epoch=len(train_loader), epochs=EPOCHS
)

# Injeksi class_weights ke CrossEntropyLoss
criterion_cls = nn.CrossEntropyLoss(weight=class_weights)
criterion_rec = nn.MSELoss()

# =========================================================
# 5. Training Loop
# =========================================================
print(f"\n🔥 Training Skenario B: Imbalance-Aware")
print(f"Metode      : SLGRAE + ENet + Class Weighting")
print(f"Device      : {device} | Threads: 12")
print("-" * 120)

best_f1 = 0.0

for epoch in range(1, EPOCHS + 1):
    start_time = time.time()
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0

    for X_b, y_b in train_loader:
        X_b, y_b = X_b.to(device, non_blocking=True), y_b.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        y_pred, x_recon, _ = model(X_b)
        
        loss = criterion_cls(y_pred, y_b) + (0.1 * criterion_rec(x_recon, X_b))
        
        loss.backward()
        optimizer.step()
        scheduler.step()

        train_loss += loss.item()
        train_correct += (y_pred.argmax(1) == y_b).sum().item()
        train_total += y_b.size(0)

    # --- VALIDATION ---
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    all_preds, all_true = [], []

    with torch.no_grad():
        for X_v, y_v in val_loader:
            X_v, y_v = X_v.to(device, non_blocking=True), y_v.to(device, non_blocking=True)
            y_v_p, x_v_r, _ = model(X_v)
            v_loss = criterion_cls(y_v_p, y_v) + (0.1 * criterion_rec(x_v_r, X_v))
            val_loss += v_loss.item()
            
            preds = y_v_p.argmax(1)
            val_correct += (preds == y_v).sum().item()
            val_total += y_v.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_true.extend(y_v.cpu().numpy())

    macro_f1 = f1_score(all_true, all_preds, average='macro')
    val_acc = 100 * val_correct / val_total
    
    # Save Best berdasarkan Macro F1 (Karena fokus kita Imbalance-Aware)
    status = ""
    if macro_f1 > best_f1:
        best_f1 = macro_f1
        torch.save({'model_state_dict': model.state_dict()}, 'best_weighted_model-skenario-b.pth')
        status = "⭐ Best F1"

    print(f"Ep [{epoch:02d}/{EPOCHS}] LR: {optimizer.param_groups[0]['lr']:.6f} | "
          f"Tr_L: {train_loss/len(train_loader):.4f} | Val_L: {val_loss/len(val_loader):.4f} | "
          f"Val_Acc: {val_acc:.2f}% | F1: {macro_f1:.4f} | {time.time()-start_time:.1f}s {status}")

print("-" * 120)
print(f"✅ Skenario B Selesai | Best Macro F1: {best_f1:.4f}")

⏳ Loading purified data & calculating weights...

🔥 Training Skenario B: Imbalance-Aware
Metode      : SLGRAE + ENet + Class Weighting
Device      : cuda | Threads: 12
------------------------------------------------------------------------------------------------------------------------
Ep [01/50] LR: 0.000050 | Tr_L: 1.9103 | Val_L: 1.2620 | Val_Acc: 77.13% | F1: 0.4787 | 20.2s ⭐ Best F1
Ep [02/50] LR: 0.000082 | Tr_L: 1.2565 | Val_L: 1.1935 | Val_Acc: 76.94% | F1: 0.4873 | 21.6s ⭐ Best F1
Ep [03/50] LR: 0.000132 | Tr_L: 1.1957 | Val_L: 1.1931 | Val_Acc: 74.54% | F1: 0.4625 | 20.4s 
Ep [04/50] LR: 0.000199 | Tr_L: 1.1621 | Val_L: 1.1527 | Val_Acc: 78.91% | F1: 0.4952 | 22.6s ⭐ Best F1
Ep [05/50] LR: 0.000280 | Tr_L: 1.1490 | Val_L: 1.1494 | Val_Acc: 76.59% | F1: 0.4821 | 20.8s 
Ep [06/50] LR: 0.000372 | Tr_L: 1.1258 | Val_L: 1.1511 | Val_Acc: 71.98% | F1: 0.4704 | 20.6s 
Ep [07/50] LR: 0.000470 | Tr_L: 1.1073 | Val_L: 1.0892 | Val_Acc: 76.68% | F1: 0.5170 | 20.2s ⭐ Best F1
Ep [08/50]

In [8]:
import torch
import torch.nn as nn
import pandas as pd
import joblib
import os
import numpy as np
import warnings # Tambahin ini buat bungkam warning gajelas
from sklearn.metrics import classification_report, accuracy_score

# =========================================================
# 0. MUTE ALL WARNINGS (Biar Output Bersih)
# =========================================================
warnings.filterwarnings("ignore")
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' 

# =========================================================
# 1. Hardware & Path Setup
# =========================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PATH_RAW_TEST = r'D:\S2\thesis\cond\project_ids\data\raw\test\test.csv'
PATH_SCALER   = r'D:\S2\thesis\cond\project_ids\data\processed\scaler_scratch.pkl'
PATH_DATA_PKL = r'D:\S2\thesis\cond\project_ids\data\processed\data_scratch.pkl'
PATH_MODEL_A  = r'D:\S2\thesis\hasil2\best_baseline_model-skenario-a.pth'
PATH_MODEL_B  = r'D:\S2\thesis\hasil2\best_weighted_model-skenario-b.pth'

# =========================================================
# 2. Arsitektur Model (Smart_SLGRAE_Latent32)
# =========================================================
class ResidualBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim), nn.BatchNorm1d(dim), nn.ReLU(),
            nn.Linear(dim, dim), nn.BatchNorm1d(dim)
        )
    def forward(self, x):
        return torch.relu(x + self.net(x))

class Smart_SLGRAE_Latent32(nn.Module):
    def __init__(self, input_dim=44, latent_dim=32, num_classes=34):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256), nn.ReLU(), ResidualBlock(256),
            nn.Linear(256, 128), nn.ReLU(), ResidualBlock(128),
            nn.Linear(128, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.ReLU(),
            nn.Linear(128, 256), nn.ReLU(),
            nn.Linear(256, input_dim)
        )
        self.classifier = nn.Sequential(
            nn.Linear(latent_dim, 256), nn.BatchNorm1d(256), nn.ReLU(),
            nn.Dropout(0.4), ResidualBlock(256),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, num_classes)
        )
    def forward(self, x):
        latent = self.encoder(x)
        return self.classifier(latent), self.decoder(latent), latent

# =========================================================
# 3. Load & Preprocess (Silently)
# =========================================================
test_df = pd.read_csv(PATH_RAW_TEST)
scaler = joblib.load(PATH_SCALER)
data_pkl = joblib.load(PATH_DATA_PKL)

training_features = scaler.feature_names_in_
class_names = data_pkl['classes']
label_map = {name: i for i, name in enumerate(class_names)}

X_raw_all = test_df.drop('label', axis=1)
y_true = np.array([label_map[str(l)] for l in test_df['label'].values])

X_raw_fixed = X_raw_all.reindex(columns=training_features, fill_value=0)
X_test_scaled = scaler.transform(X_raw_fixed)
X_tensor = torch.tensor(X_test_scaled).float().to(device)

# =========================================================
# 4. Fungsi Evaluasi (Clean Output)
# =========================================================
def run_final_test(model_path, scenario_name):
    print(f"\n{'='*70}")
    print(f" RESULT: {scenario_name}")
    print(f"{'='*70}")
    
    model = Smart_SLGRAE_Latent32().to(device)
    checkpoint = torch.load(model_path, map_location=device)
    
    if 'model_state_dict' in checkpoint:
        model.load_state_dict(checkpoint['model_state_dict'])
    else:
        model.load_state_dict(checkpoint)
        
    model.eval()
    with torch.no_grad():
        y_pred_logits, _, _ = model(X_tensor)
        y_pred_final = torch.max(y_pred_logits, 1)[1].cpu().numpy()
    
    acc = accuracy_score(y_true, y_pred_final)
    print(f"Overall Accuracy : {acc:.4f}")
    print("-" * 70)
    print(classification_report(y_true, y_pred_final, target_names=class_names, digits=4))

# =========================================================
# 5. Eksekusi
# =========================================================
if os.path.exists(PATH_MODEL_A):
    run_final_test(PATH_MODEL_A, "SKENARIO A (BASELINE)")

if os.path.exists(PATH_MODEL_B):
    run_final_test(PATH_MODEL_B, "SKENARIO B (IMBALANCE-AWARE)")


 RESULT: SKENARIO A (BASELINE)
Overall Accuracy : 0.9920
----------------------------------------------------------------------
                         precision    recall  f1-score   support

       Backdoor_Malware     0.6000    0.0337    0.0638        89
          BenignTraffic     0.8808    0.9790    0.9273     27709
       BrowserHijacking     0.7200    0.1343    0.2264       134
       CommandInjection     0.4485    0.5126    0.4784       119
 DDoS-ACK_Fragmentation     0.9891    0.9936    0.9913      7292
        DDoS-HTTP_Flood     0.9766    0.9436    0.9598       709
        DDoS-ICMP_Flood     0.9996    0.9999    0.9997    180447
DDoS-ICMP_Fragmentation     0.9943    0.9968    0.9955     11402
      DDoS-PSHACK_Flood     0.9995    0.9997    0.9996    103326
       DDoS-RSTFINFlood     0.9999    0.9994    0.9997    101819
         DDoS-SYN_Flood     0.9982    0.9994    0.9988    102208
         DDoS-SlowLoris     0.9192    0.9325    0.9258       622
DDoS-SynonymousIP_Flood  